In [10]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display
#オブジェクト作成
Arm = Arm_Device()
time.sleep(.1)

In [2]:
#各サーボの角度を取得
for id in range(6):
    arm_position=Arm.Arm_serial_servo_read_any(id+1)
    print(f'{id+1} arm position {arm_position}')

1 arm position 89
2 arm position 90
3 arm position 90
4 arm position 90
5 arm position 48
6 arm position 90


In [3]:
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), # 0: 地面
    
    ikpy.link.URDFLink(
        name="id1_base_turn",   # 1: サーボID 1 (台座)
        origin_translation=np.array([0.0, 0.0, 0.0675]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加：初期の傾き（0度）
        rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="id2_shoulder",    # 2: サーボID 2 (肩)
        origin_translation=np.array([0.0, 0.0, 0.040]),  
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="id3_elbow",       # 3: サーボID 3 (肘)
        origin_translation=np.array([0.0, 0.0, 0.08285]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))
    ),
    ikpy.link.URDFLink(
        name="id4_wrist_pitch", # 4: サーボID 4 (手首上下)
        origin_translation=np.array([0.0, 0.0, 0.08285]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))
    ),
    ikpy.link.URDFLink(
        name="id5_wrist_roll",  # 5: サーボID 5 (手首ひねり)
        origin_translation=np.array([0.0, 0.0, 0.07905]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))
    ),
    ikpy.link.URDFLink(
        name="gripper_tip",     # 6: ハサミの先端（ダミー）
        origin_translation=np.array([0.0, 0.0, 0.1203]),  
        origin_orientation=np.array([0.0, 0.0, 0.0]), # ★追加
        rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0)
    )
], active_links_mask=[False, True, True, True, True, True, False])

In [23]:
def cal_servo_angles(target_x,target_y,target_z):
    # =====================================================================
    # 2. 動かしたい「目標の場所（座標）」を決める
    # =====================================================================
    # 例：ロボットの正面「まっすぐ前方に15cm、高さ25cm」の位置へ移動させたい場合
    target_position = np.array([target_x, target_y, target_z])
    
    # 手先の向き：ハサミを「真下」に向ける（ピッチ90度）
    pitch = np.radians(90)
    target_orientation = np.array([
    [np.cos(pitch),  0.0, np.sin(pitch)],
    [0.0,            1.0, 0.0          ],
    [-np.sin(pitch), 0.0, np.cos(pitch)]
    ])
    
    # 4x4の目標行列（ターゲットフレーム）を作成
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    # =====================================================================
    # 3. 逆運動学の計算をスタート
    # =====================================================================
    # 初期姿勢（垂直ポーズ）から計算を開始する
    # initial_angles = [0.0] * len(dofbot_chain.links)
   
    # IKの実行（目標ポーズに届く各関節の角度をラジアンで一瞬で計算）
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=computed_angles)
    
    # =====================================================================
    # 4. 計算結果を実機用の角度に直して表示
    # =====================================================================
    # print("=========================================")
    # print("          Dofbot IK 計算完了             ")
    # print("=========================================")
    # print(f"目標座標 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm\n")
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} # 実機命令用の角度を保存する辞書
    
    # index 1〜5（可動する5つのサーボ）の結果を取り出す
    for i, angle_rad in enumerate(computed_angles[1:6]):
        angle_deg = np.degrees(angle_rad)
        
        # 90度を足して、実機サーボ用の命令角度（0度〜180度）に変換
        dofbot_servo_angle = 90.0 + angle_deg
        
        # 小数点第1位までに丸めて保存・表示
        target_angles[servo_ids[i]] = round(dofbot_servo_angle, 1)
        # print(f"サーボ ID {servo_ids[i]} ➔  {target_angles[servo_ids[i]]:.1f} °")
        
    # print("=========================================")
    return target_angles
    
    # Arm.Arm_serial_servo_write6(target_angles[1], target_angles[2], target_angles[3], target_angles[4], target_angles[5], 90, 5000)
    # time.sleep(2)
    
    
    # # =====================================================================
    # # 5. 【参考】実際に実機モータを動かす場合のコード例
    # # =====================================================================
    # # ※ 実際にDofbot（Jetson Nano等）の上で動かす時は、以下のコメントアウトを外します。
    # """
    # from Arm_Lib import Arm_Device
    # import time
    
    # Arm = Arm_Device() # ロボットアームの制御オブジェクトを用意
    
    # # 計算された角度を、すべてのサーボに同時に送信して1秒(1000ms)かけて動かす
    # # Dofbot公式ライブラリの関数の引数は、(ID, 角度, 動かす時間) です。
    # # ※注意：実機の基盤のピン番号（ID）と、ここで計算したIDが一致している必要があります。
    # Arm.Arm_Serial_Servo_Write(1, target_angles[1], 1000)
    # Arm.Arm_Serial_Servo_Write(2, target_angles[2], 1000)
    # Arm.Arm_Serial_Servo_Write(3, target_angles[3], 1000)
    # Arm.Arm_Serial_Servo_Write(4, target_angles[4], 1000)
    # Arm.Arm_Serial_Servo_Write(5, target_angles[5], 1000)
    # time.sleep(1) # 動き終わるまで1秒待つ
    # """

In [17]:
target_x=0
target_y=0
target_z=472.55
angles = cal_servo_angles(target_x,target_y,target_z)
for i in range(5):
    print(angles[i+1])

90.0
90.0
90.0
90.0
90.0


In [24]:
target_x = 0.0
target_y = 0.0
target_z = 0.47255
STEP = 0.01
DURATION = 300    # アームの移動時間（300ミリ秒 = 0.3秒）
WAIT_TIME = 0.3   # Python側の待ち時間（0.3秒）

out_b = widgets.Output(layout=widgets.Layout(height='180px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のテキストボックスをクリックして W / A / S / D を入力してください。")

key_receiver = widgets.Text(
    value='',
    placeholder='ここにフォーカスを合わせてキー入力',
    layout=widgets.Layout(width='320px')
)

app_b = widgets.VBox([label_desc, key_receiver, out_b])

# ==========================================
# 2. テキストの変更を検知する関数
# ==========================================
def handle_text_change(change):
    global target_x, target_y, target_z
    
    raw_val = change['new']
    if not raw_val:
        return # 自動クリアによる空文字（""）への変化は無視する
        
    key = raw_val[-1].lower()
    
    with out_b:
        # ★ポイント1：前の出力を消去して、常に最新情報だけを1箇所に表示する
        out_b.clear_output(wait=True)
        
        # ★ポイント2：各キーに応じて座標を増減させる
        if key == 'w':
            target_x -= STEP
            print(f"【W】前進（Xを {STEP} 加算）")
        elif key == 's':
            target_x += STEP
            print(f"【S】後退（Xを {STEP} 減算）")
        elif key == 'a':
            target_y -= STEP
            print(f"【A】左スライド（Yを {STEP} 減算）")
        elif key == 'd':
            target_y += STEP
            print(f"【D】右スライド（Yを {STEP} 加算）")
        elif key == 'f':
            target_z += STEP
            print(f"【A】左スライド（Yを {STEP} 減算）")
        elif key == 'r':
            target_z -= STEP
            print(f"【D】右スライド（Yを {STEP} 加算）")
        elif key == 'q':
            print("【Q】セッションを終了しウィジェットを閉じます。")
            app_b.close()
            return
        else:
            print(f"未対応のキーです: {key}")
            # 対応外のキーの場合は、以下の計算をスキップして入力欄をクリアする
            key_receiver.value = ''
            return

        # 座標が更新されたので、サーボの角度を再計算して表示
        try:
            angles = cal_servo_angles(target_x, target_y, target_z)
            print("-" * 30)
            print(f"現在の座標 -> X: {target_x:.2f}, Y: {target_y:.2f}, Z: {target_z:.2f}")
            print("-" * 30)
            print("計算されたサーボ角度:")
            for i in range(5):
                print(f"  サーボ {i+1}: {angles[i+1]} 度")
            Arm.Arm_serial_servo_write6(
                angles[1], angles[2], angles[3], angles[4], angles[5], 90, DURATION
            )
            
            # 4. ★ アームが動き終わるまで待つ（2秒からWAIT_TIME(0.3秒)に変更）
            time.sleep(WAIT_TIME)
        except Exception as e:
            print(f"計算エラーが発生しました: {e}")

    # 入力された値を消去して、次の入力を待ち受ける
    key_receiver.value = ''

# 3. 監視設定と表示
key_receiver.observe(handle_text_change, names='value')
print("【WASD座標制御モード】監視スタート")
display(app_b)

【WASD座標制御モード】監視スタート


In [29]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display

# =====================================================================
# 1. オブジェクト作成と初期化
# =====================================================================
Arm = Arm_Device()
time.sleep(.1)

# ロボットの構造定義（台座=ID1, 爪=ID6 の決定版寸法）
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), 
    ikpy.link.URDFLink(name="id1_base_turn",   origin_translation=np.array([0.0, 0.0, 0.0675]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="id2_shoulder",    origin_translation=np.array([0.0, 0.0, 0.040]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="id3_elbow",       origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))),
    ikpy.link.URDFLink(name="id4_wrist_pitch", origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))),
    ikpy.link.URDFLink(name="id5_wrist_roll",  origin_translation=np.array([0.0, 0.0, 0.07905]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="gripper_tip",     origin_translation=np.array([0.0, 0.0, 0.1203]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0))
], active_links_mask=[False, True, True, True, True, True, False])

# =====================================================================
# 2. グローバル変数と【安全壁】の定義
# =====================================================================
target_x = -0.15   
target_y = 0.0    
target_z = 0.25   

STEP = 0.01
DURATION = 200    # 移動時間を200msに縮めてキビキビ感をアップ
WAIT_TIME = 0.2   # 待ち時間も合わせて短縮して大渋滞を防ぐ

computed_angles = [0.0] * len(dofbot_chain.links)

# ★直前のサーボ角度（度）を覚えておくための変数
last_servo_angles = [90.0, 90.0, 90.0, 90.0, 90.0]

# ★作戦1：セーフティ・ボックス（安全な移動範囲の壁：単位メートル）
LIMIT_X_MIN, LIMIT_X_MAX = 0.08, 0.24   # 前後の限界（引き寄せすぎ・伸びきり防止）
LIMIT_Y_MIN, LIMIT_Y_MAX = -0.15, 0.15  # 左右の限界
LIMIT_Z_MIN, LIMIT_Z_MAX = 0.08, 0.35   # 高さの限界（地面衝突・突っ張り防止）

# =====================================================================
# 3. 逆運動学の計算関数
# =====================================================================
def cal_servo_angles(target_x, target_y, target_z):
    global computed_angles
    
    target_position = np.array([target_x, target_y, target_z])
    pitch = np.radians(90) 
    target_orientation = np.array([
        [np.cos(pitch),  0.0, np.sin(pitch)],
        [0.0,            1.0, 0.0          ],
        [-np.sin(pitch), 0.0, np.cos(pitch)]
    ])
    
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=computed_angles)
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} 
    for i, angle_rad in enumerate(computed_angles[1:6]):
        angle_deg = np.degrees(angle_rad)
        dofbot_servo_angle = 90.0 + angle_deg
        target_angles[servo_ids[i]] = round(dofbot_servo_angle, 1)
        
    return target_angles

# =====================================================================
# 4. UIウィジェットの構築
# =====================================================================
out_b = widgets.Output(layout=widgets.Layout(height='220px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のボックスをクリックして W/A/S/D/R/F を入力（Qで終了）")
key_receiver = widgets.Text(value='', placeholder='ここにフォーカスを合わせてキー入力', layout=widgets.Layout(width='320px'))
app_b = widgets.VBox([label_desc, key_receiver, out_b])

# =====================================================================
# 5. コールバック関数
# =====================================================================
def handle_text_change(change):
    global target_x, target_y, target_z, last_servo_angles
    
    raw_val = change['new']
    if not raw_val:
        return 
        
    key = raw_val[-1].lower()
    
    with out_b:
        out_b.clear_output(wait=True)
        
        # キー入力による座標変更
        if key == 'w':   target_x += STEP
        elif key == 's': target_x -= STEP
        elif key == 'a': target_y -= STEP
        elif key == 'd': target_y += STEP
        elif key == 'r': target_z += STEP
        elif key == 'f': target_z -= STEP
        elif key == 'q':
            print("【Q】終了します。")
            app_b.close()
            return
        else:
            print(f"未対応のキー: {key}")
            key_receiver.value = ''
            return

        # ★作戦1：セーフティ・ボックス（安全壁）の適用
        # np.clip(値, 最小, 最大) を使って、範囲外の数値を自動で壁に張り付けます
        target_x = np.clip(target_x, LIMIT_X_MIN, LIMIT_X_MAX)
        target_y = np.clip(target_y, LIMIT_Y_MIN, LIMIT_Y_MAX)
        target_z = np.clip(target_z, LIMIT_Z_MIN, LIMIT_Z_MAX)

        try:
            # 角度計算
            angles = cal_servo_angles(target_x, target_y, target_z)
            new_angles = [angles[1], angles[2], angles[3], angles[4], angles[5]]
            
            # ★作戦2：急ブレーキ（角度ジャンプ・フィルター）
            # 1回のキー入力で、いずれかのサーボが「25度以上」跳ぼうとしたら暴走と判定
            MAX_JUMP = 25.0
            is_valid = True
            
            for i in range(5):
                diff = abs(new_angles[i] - last_servo_angles[i])
                if diff > MAX_JUMP:
                    print(f"⚠️ [安全ストップ] サーボ{i+1}が急激に動こうとしました(差分:{diff:.1f}°)。命令をブロックします。")
                    is_valid = False
                    break
            
            if is_valid:
                # 暴走の危険がなければ、実機に送信し、現在の角度を記憶する
                print(f"目標座標 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm")
                print(f"送信角度 ➔ ID1:{new_angles[0]:.1f}°, ID2:{new_angles[1]:.1f}°, ID3:{new_angles[2]:.1f}°, ID4:{new_angles[3]:.1f}°, ID5:{new_angles[4]:.1f}°")
                
                Arm.Arm_serial_servo_write6(
                    new_angles[0], new_angles[1], new_angles[2], new_angles[3], new_angles[4], 90, DURATION
                )
                last_servo_angles = new_angles  # 正常な角度として記憶を更新
                time.sleep(WAIT_TIME)
            else:
                # 危険な座標だった場合は、内部の座標変数を一歩手前に戻す（これ以上進ませない）
                if key == 'w':   target_x -= STEP
                elif key == 's': target_x += STEP
                elif key == 'a': target_y += STEP
                elif key == 'd': target_y -= STEP
                elif key == 'r': target_z -= STEP
                elif key == 'f': target_z += STEP

        except Exception as e:
            print(f"計算エラー: {e}")

    key_receiver.value = ''

# =====================================================================
# 6. 起動時の自動ポジショニング と 表示
# =====================================================================
key_receiver.observe(handle_text_change, names='value')
print("【安全ガード付き WASD+RFモード】監視スタート")
print("➔ 実機アームを操作初期位置に移動中...")

try:
    initial_angles = cal_servo_angles(target_x, target_y, target_z)
    last_servo_angles = [initial_angles[1], initial_angles[2], initial_angles[3], initial_angles[4], initial_angles[5]]
    Arm.Arm_serial_servo_write6(
        last_servo_angles[0], last_servo_angles[1], last_servo_angles[2], 
        last_servo_angles[3], last_servo_angles[4], 90, 1500
    )
    time.sleep(1.5)
    print("➔ 連動完了！安全に操作できます。")
except Exception as e:
    print(f"初期移動エラー: {e}")

display(app_b)

【安全ガード付き WASD+RFモード】監視スタート
➔ 実機アームを操作初期位置に移動中...
➔ 連動完了！安全に操作できます。


In [30]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display

# =====================================================================
# 1. オブジェクト作成と初期化
# =====================================================================
Arm = Arm_Device()
time.sleep(.1)

# ロボットの構造定義
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), 
    ikpy.link.URDFLink(
        name="id1_base_turn",   
        origin_translation=np.array([0.0, 0.0, 0.0675]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]),
        rotation=np.array([0.0, 0.0, 1.0]), 
        bounds=(np.radians(-180), np.radians(180)) # ★後ろに振り向けるように ±180度 に拡大
    ),
    ikpy.link.URDFLink(name="id2_shoulder",    origin_translation=np.array([0.0, 0.0, 0.040]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="id3_elbow",       origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))),
    ikpy.link.URDFLink(name="id4_wrist_pitch", origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))),
    ikpy.link.URDFLink(name="id5_wrist_roll",  origin_translation=np.array([0.0, 0.0, 0.07905]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="gripper_tip",     origin_translation=np.array([0.0, 0.0, 0.1203]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0))
], active_links_mask=[False, True, True, True, True, True, False])

# =====================================================================
# 2. グローバル変数と【安全壁】の定義
# =====================================================================
# ★ご要望通り、Xをマイナス（後ろ側15cm）からスタート
target_x = -0.15   
target_y = 0.0    
target_z = 0.25   

STEP = 0.01
DURATION = 200    
WAIT_TIME = 0.2   

# 角度記憶変数の用意
computed_angles = [0.0] * len(dofbot_chain.links)
last_servo_angles = [90.0, 90.0, 90.0, 90.0, 90.0]

# ★作戦1：セーフティ・ボックス（Xをマイナス範囲まで広げました）
LIMIT_X_MIN, LIMIT_X_MAX = -0.24, 0.24  # 後ろ24cm 〜 前24cm まで移動可能
LIMIT_Y_MIN, LIMIT_Y_MAX = -0.15, 0.15  
LIMIT_Z_MIN, LIMIT_Z_MAX = 0.08, 0.35   

# =====================================================================
# 3. 逆運動学の計算関数
# =====================================================================
def cal_servo_angles(target_x, target_y, target_z, base_angles=None):
    global computed_angles
    
    target_position = np.array([target_x, target_y, target_z])
    pitch = np.radians(90) 
    target_orientation = np.array([
        [np.cos(pitch),  0.0, np.sin(pitch)],
        [0.0,            1.0, 0.0          ],
        [-np.sin(pitch), 0.0, np.cos(pitch)]
    ])
    
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    # 計算のヒントにする初期姿勢を指定
    start_angles = base_angles if base_angles is not None else computed_angles
    
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=start_angles)
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} 
    for i, angle_rad in enumerate(computed_angles[1:6]):
        angle_deg = np.degrees(angle_rad)
        target_angles[servo_ids[i]] = round(90.0 + angle_deg, 1)
        
    return target_angles

# =====================================================================
# 4. UIウィジェットの構築
# =====================================================================
out_b = widgets.Output(layout=widgets.Layout(height='220px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のボックスをクリックして W/A/S/D/R/F を入力（Qで終了）")
key_receiver = widgets.Text(value='', placeholder='ここにフォーカスを合わせてキー入力', layout=widgets.Layout(width='320px'))
app_b = widgets.VBox([label_desc, key_receiver, out_b])

# =====================================================================
# 5. コールバック関数
# =====================================================================
def handle_text_change(change):
    global target_x, target_y, target_z, last_servo_angles, computed_angles
    
    raw_val = change['new']
    if not raw_val:
        return 
        
    key = raw_val[-1].lower()
    
    with out_b:
        out_b.clear_output(wait=True)
        
        # ★バグ修正：WとSの「加算・減算」がごっちゃになっていたのを、直感に合わせて修正
        if key == 'w':   target_x += STEP  # 前に進む
        elif key == 's': target_x -= STEP  # 後ろに下がる
        elif key == 'a': target_y -= STEP  # 左
        elif key == 'd': target_y += STEP  # 右
        elif key == 'r': target_z += STEP  # 上
        elif key == 'f': target_z -= STEP  # 下
        elif key == 'q':
            print("【Q】終了します。")
            app_b.close()
            return
        else:
            print(f"未対応のキー: {key}")
            key_receiver.value = ''
            return

        # セーフティ・ボックス壁の適用
        target_x = np.clip(target_x, LIMIT_X_MIN, LIMIT_X_MAX)
        target_y = np.clip(target_y, LIMIT_Y_MIN, LIMIT_Y_MAX)
        target_z = np.clip(target_z, LIMIT_Z_MIN, LIMIT_Z_MAX)

        try:
            # 判別用に、「計算前の状態」を一時キープしておく（ロールバック用）
            old_computed_angles = np.copy(computed_angles)
            
            # 角度計算
            angles = cal_servo_angles(target_x, target_y, target_z)
            new_angles = [angles[1], angles[2], angles[3], angles[4], angles[5]]
            
            # 作戦2：角度ジャンプ・フィルター（許容度を少し広げて30度に調整）
            MAX_JUMP = 30.0
            is_valid = True
            
            for i in range(5):
                diff = abs(new_angles[i] - last_servo_angles[i])
                if diff > MAX_JUMP:
                    print(f"⚠️ [安全ストップ] サーボ{i+1}が急激に動こうとしました(差分:{diff:.1f}°)。命令をガードしました。")
                    is_valid = False
                    break
            
            if is_valid:
                # 安全なら実機を動かし、記憶を更新
                print(f"目標座標 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm")
                print(f"送信角度 ➔ ID1:{new_angles[0]:.1f}°, ID2:{new_angles[1]:.1f}°, ID3:{new_angles[2]:.1f}°, ID4:{new_angles[3]:.1f}°, ID5:{new_angles[4]:.1f}°")
                
                Arm.Arm_serial_servo_write6(
                    new_angles[0], new_angles[1], new_angles[2], new_angles[3], new_angles[4], 90, DURATION
                )
                last_servo_angles = new_angles  
                time.sleep(WAIT_TIME)
            else:
                # ❌ 計算が跳んでブロックされた場合は、プログラムの記憶を「計算前」に巻き戻す！
                # これにより無限ロックバグを完全に回避します
                computed_angles = old_computed_angles
                
                # 座標データも一歩手前に戻す
                if key == 'w':   target_x -= STEP
                elif key == 's': target_x += STEP
                elif key == 'a': target_y += STEP
                elif key == 'd': target_y -= STEP
                elif key == 'r': target_z -= STEP
                elif key == 'f': target_z += STEP

        except Exception as e:
            print(f"計算エラー: {e}")

    key_receiver.value = ''

# =====================================================================
# 6. 起動時の自動ポジショニング と 表示
# =====================================================================
key_receiver.observe(handle_text_change, names='value')
print("【無限ロック解消版・安全ガード付き WASD+RFモード】監視スタート")
print(f"➔ 実機アームを初期位置 [後ろ側] (X:{target_x*100:.1f}cm, Z:{target_z*100:.1f}cm) に移動中...")

try:
    # 起動時は、真っ直格好 [0.0]*7 から確実に「後ろ側の解」を見つけさせます
    initial_angles = cal_servo_angles(target_x, target_y, target_z, base_angles=[0.0]*7)
    last_servo_angles = [initial_angles[1], initial_angles[2], initial_angles[3], initial_angles[4], initial_angles[5]]
    
    # 1.5秒かけて後ろ側のポジションに移動
    Arm.Arm_serial_servo_write6(
        last_servo_angles[0], last_servo_angles[1], last_servo_angles[2], 
        last_servo_angles[3], last_servo_angles[4], 90, 1500
    )
    time.sleep(1.5)
    print("➔ 後ろ向きでの連動完了！安全に操作できます。")
except Exception as e:
    print(f"初期移動エラー: {e}")

display(app_b)

【無限ロック解消版・安全ガード付き WASD+RFモード】監視スタート
➔ 実機アームを初期位置 [後ろ側] (X:-15.0cm, Z:25.0cm) に移動中...
➔ 後ろ向きでの連動完了！安全に操作できます。


In [36]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display

# =====================================================================
# 1. オブジェクト作成と初期化
# =====================================================================
Arm = Arm_Device()

# ロボットの構造定義
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), 
    ikpy.link.URDFLink(
        name="id1_base_turn",   
        origin_translation=np.array([0.0, 0.0, 0.0675]), 
        origin_orientation=np.array([0.0, 0.0, 0.0]),
        rotation=np.array([0.0, 0.0, 1.0]), 
        bounds=(np.radians(-180), np.radians(180)) # 後ろに振り向けるように ±180度
    ),
    ikpy.link.URDFLink(name="id2_shoulder",    origin_translation=np.array([0.0, 0.0, 0.040]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="id3_elbow",       origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))),
    ikpy.link.URDFLink(name="id4_wrist_pitch", origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))),
    ikpy.link.URDFLink(name="id5_wrist_roll",  origin_translation=np.array([0.0, 0.0, 0.07905]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="gripper_tip",     origin_translation=np.array([0.0, 0.0, 0.1203]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0))
], active_links_mask=[False, True, True, True, True, True, False])

# =====================================================================
# 2. グローバル変数と【安全壁】の定義
# =====================================================================
# Xをマイナス（後ろ側15cm）からスタート
target_x = -0.15   
target_y = 0.0    
target_z = 0.25   

STEP = 0.01
DURATION = 500    
WAIT_TIME = 0.5   

computed_angles = [0.0] * len(dofbot_chain.links)
last_servo_angles = [90.0, 90.0, 90.0, 90.0, 90.0]

# セーフティ・ボックス（大まかなガード）
LIMIT_X_MIN, LIMIT_X_MAX = -0.24, 0.24  
LIMIT_Y_MIN, LIMIT_Y_MAX = -0.15, 0.15  
LIMIT_Z_MIN, LIMIT_Z_MAX = 0.08, 0.35   

# =====================================================================
# 3. 逆運動学の計算関数
# =====================================================================
def cal_servo_angles(target_x, target_y, target_z, base_angles=None):
    global computed_angles
    
    target_position = np.array([target_x, target_y, target_z])
    
    # ★修正の核心：ハサミを【地球の地面に対して常に真下】に向ける正しい回転行列
    target_orientation = np.array([
        [-1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0,  0.0, -1.0]
    ])
    
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    start_angles = base_angles if base_angles is not None else computed_angles
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=start_angles)
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} 
    for i, angle_rad in enumerate(computed_angles[1:6]):
        target_angles[servo_ids[i]] = round(90.0 + np.degrees(angle_rad), 1)
        
    return target_angles

# =====================================================================
# 4. UIウィジェットの構築
# =====================================================================
out_b = widgets.Output(layout=widgets.Layout(height='220px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のボックスをクリックして W/A/S/D/R/F を入力（Qで終了）")
key_receiver = widgets.Text(value='', placeholder='ここにフォーカスを合わせてキー入力', layout=widgets.Layout(width='320px'))
app_b = widgets.VBox([label_desc, key_receiver, out_b])

# =====================================================================
# 5. コールバック関数（★ユーザーのアイデアを具現化）
# =====================================================================
def handle_text_change(change):
    global target_x, target_y, target_z, last_servo_angles, computed_angles
    
    raw_val = change['new']
    if not raw_val:
        return 
        
    key = raw_val[-1].lower()
    
    with out_b:
        out_b.clear_output(wait=True)
        
        # 1. まずは「次の候補の座標（1cm先）」を仮作成する
        next_x, next_y, next_z = target_x, target_y, target_z
        if key == 'w':   next_x += STEP  
        elif key == 's': next_x -= STEP  
        elif key == 'a': next_y -= STEP  
        elif key == 'd': next_y += STEP  
        elif key == 'r': next_z += STEP  
        elif key == 'f': next_z -= STEP  
        elif key == 'q':
            print("【Q】終了します。")
            app_b.close()
            return
        else:
            print(f"未対応のキー: {key}")
            key_receiver.value = ''
            return

        # 大まかな安全壁の適用
        next_x = np.clip(next_x, LIMIT_X_MIN, LIMIT_X_MAX)
        next_y = np.clip(next_y, LIMIT_Y_MIN, LIMIT_Y_MAX)
        next_z = np.clip(next_z, LIMIT_Z_MIN, LIMIT_Z_MAX)

        try:
            # 計算前の状態を一時的にバックアップ（不合格のときに巻き戻すため）
            old_computed_angles = np.copy(computed_angles)
            
            # 正しい「真下固定」の回転行列でIKを計算してみる
            target_position = np.array([next_x, next_y, next_z])
            target_orientation = np.array([
                [-1.0,  0.0,  0.0],
                [ 0.0,  1.0,  0.0],
                [ 0.0,  0.0, -1.0]
            ])
            target_frame = np.eye(4)
            target_frame[:3, :3] = target_orientation
            target_frame[:3, 3] = target_position
            
            # 投機的実行（一度計算させてみる）
            computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=old_computed_angles)
            
            # 2. ★【答え合わせ】順運動学（FK）を使って、本当に手が届いているか検証！
            real_frame = dofbot_chain.forward_kinematics(computed_angles)
            real_position = real_frame[:3, 3]
            
            # 指示した座標と、計算上本当に届く座標の間の「ズレの直線距離」
            distance_error = np.linalg.norm(real_position - target_position)
            
            # サーボ角度のジャンプチェック
            new_angles = [90.0 + np.degrees(ang) for ang in computed_angles[1:6]]
            MAX_JUMP = 30.0
            jump_ok = True
            for i in range(5):
                if abs(new_angles[i] - last_servo_angles[i]) > MAX_JUMP:
                    jump_ok = False
                    break
            
            # 3. ★【判定】ズレが1.5cm未満、かつ角度の跳びがない（＝実現可能）なら正式採用！
            if distance_error < 0.015 and jump_ok:
                target_x, target_y, target_z = next_x, next_y, next_z
                last_servo_angles = new_angles
                
                print(f"現在の位置 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm")
                print(f"送信角度 ➔ ID1:{new_angles[0]:.1f}°, ID2:{new_angles[1]:.1f}°, ID3:{new_angles[2]:.1f}°, ID4:{new_angles[3]:.1f}°, ID5:{new_angles[4]:.1f}°")
                
                Arm.Arm_serial_servo_write6(
                    new_angles[0], new_angles[1], new_angles[2], new_angles[3], new_angles[4], 90, DURATION
                )
                time.sleep(WAIT_TIME)
            else:
                # 4. ❌【却下】実現不可能な値なら、一歩手前の位置に踏みとどまる！
                computed_angles = old_computed_angles # 記憶を過去に巻き戻す
                print(f"🛑 [境界線キープ] その位置は届かない（または姿勢が壊れる）ため、安全な限界位置で停止しました。")

        except Exception as e:
            print(f"計算エラー: {e}")

    key_receiver.value = ''

# =====================================================================
# 6. 起動時の自動ポジショニング と 表示
# =====================================================================
key_receiver.observe(handle_text_change, names='value')
print("【限界自動キープ機能つき WASD+RFモード】監視スタート")
print(f"➔ 実機アームを初期位置 [後ろ側] (X:{target_x*100:.1f}cm, Z:{target_z*100:.1f}cm) に移動中...")

try:
    initial_angles = cal_servo_angles(target_x, target_y, target_z, base_angles=[0.0]*7)
    last_servo_angles = [initial_angles[1], initial_angles[2], initial_angles[3], initial_angles[4], initial_angles[5]]
    
    Arm.Arm_serial_servo_write6(
        last_servo_angles[0], last_servo_angles[1], last_servo_angles[2], 
        last_servo_angles[3], last_servo_angles[4], 90, 1500
    )
    time.sleep(1.5)
    print("➔ 連動完了！安全に操作できます。")
except Exception as e:
    print(f"初期移動エラー: {e}")

display(app_b)

【限界自動キープ機能つき WASD+RFモード】監視スタート
➔ 実機アームを初期位置 [後ろ側] (X:-15.0cm, Z:25.0cm) に移動中...
➔ 連動完了！安全に操作できます。


In [1]:
#!/usr/bin/env python3
#coding=utf-8
import time
from Arm_Lib import Arm_Device
import numpy as np
import ikpy.chain
import ikpy.link
import ipywidgets as widgets
from IPython.display import display

# =====================================================================
# 1. オブジェクト作成と初期化
# =====================================================================
Arm = Arm_Device()
time.sleep(.1)

# ロボットの構造定義
dofbot_chain = ikpy.chain.Chain(links=[
    ikpy.link.OriginLink(), 
    ikpy.link.URDFLink(name="id1_base_turn",   origin_translation=np.array([0.0, 0.0, 0.0675]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-180), np.radians(180))),
    ikpy.link.URDFLink(name="id2_shoulder",    origin_translation=np.array([0.0, 0.0, 0.040]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="id3_elbow",       origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-80), np.radians(80))),
    ikpy.link.URDFLink(name="id4_wrist_pitch", origin_translation=np.array([0.0, 0.0, 0.08285]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 1.0, 0.0]), bounds=(np.radians(-75), np.radians(75))),
    ikpy.link.URDFLink(name="id5_wrist_roll",  origin_translation=np.array([0.0, 0.0, 0.07905]), origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 1.0]), bounds=(np.radians(-90), np.radians(90))),
    ikpy.link.URDFLink(name="gripper_tip",     origin_translation=np.array([0.0, 0.0, 0.1203]),  origin_orientation=np.array([0.0, 0.0, 0.0]), rotation=np.array([0.0, 0.0, 0.0]), bounds=(0, 0))
], active_links_mask=[False, True, True, True, True, True, False])

# =====================================================================
# 2. グローバル変数と【安全壁】の定義
# =====================================================================
target_x = -0.15   
target_y = 0.0    
target_z = 0.25   

STEP = 0.01
DURATION = 200    
WAIT_TIME = 0.2   

computed_angles = [0.0] * len(dofbot_chain.links)

# ★角度を記憶する変数（最新ポーズ と ワンステップ前のポーズ）
last_servo_angles = [90.0, 90.0, 90.0, 90.0, 90.0]  # 現在（最新）の角度
prev_servo_angles = [90.0, 90.0, 90.0, 90.0, 90.0]  # ★追加：1つ前の角度

# セーフティ・ボックス
LIMIT_X_MIN, LIMIT_X_MAX = -0.24, 0.24  
LIMIT_Y_MIN, LIMIT_Y_MAX = -0.15, 0.15  
LIMIT_Z_MIN, LIMIT_Z_MAX = 0.08, 0.35   

# =====================================================================
# 3. 逆運動学の計算関数
# =====================================================================
def cal_servo_angles(target_x, target_y, target_z, base_angles=None):
    global computed_angles
    
    target_position = np.array([target_x, target_y, target_z])
    target_orientation = np.array([
        [-1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0,  0.0, -1.0]
    ])
    
    target_frame = np.eye(4)
    target_frame[:3, :3] = target_orientation
    target_frame[:3, 3] = target_position
    
    start_angles = base_angles if base_angles is not None else computed_angles
    computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=start_angles)
    
    servo_ids = [1, 2, 3, 4, 5]
    target_angles = {} 
    for i, angle_rad in enumerate(computed_angles[1:6]):
        target_angles[servo_ids[i]] = round(90.0 + np.degrees(angle_rad), 1)
        
    return target_angles

# =====================================================================
# 4. UIウィジェットの構築
# =====================================================================
out_b = widgets.Output(layout=widgets.Layout(height='260px', border='1px solid #cbd5e1', overflow_y='auto'))
label_desc = widgets.HTML("<b>キー監視入力欄：</b> 下のボックスをクリックして W/A/S/D/R/F を入力（Qで終了）")
key_receiver = widgets.Text(value='', placeholder='ここにフォーカスを合わせてキー入力', layout=widgets.Layout(width='320px'))
app_b = widgets.VBox([label_desc, key_receiver, out_b])

# =====================================================================
# 5. コールバック関数
# =====================================================================
def handle_text_change(change):
    # global宣言に prev_servo_angles を追加
    global target_x, target_y, target_z, last_servo_angles, prev_servo_angles, computed_angles
    
    raw_val = change['new']
    if not raw_val:
        return 
        
    key = raw_val[-1].lower()
    
    with out_b:
        out_b.clear_output(wait=True)
        
        next_x, next_y, next_z = target_x, target_y, target_z
        if key == 'w':   next_x += STEP  
        elif key == 's': next_x -= STEP  
        elif key == 'a': next_y -= STEP  
        elif key == 'd': next_y += STEP  
        elif key == 'r': next_z += STEP  
        elif key == 'f': next_z -= STEP  
        elif key == 'q':
            print("【Q】終了します。")
            app_b.close()
            return
        else:
            print(f"未対応のキー: {key}")
            key_receiver.value = ''
            return

        next_x = np.clip(next_x, LIMIT_X_MIN, LIMIT_X_MAX)
        next_y = np.clip(next_y, LIMIT_Y_MIN, LIMIT_Y_MAX)
        next_z = np.clip(next_z, LIMIT_Z_MIN, LIMIT_Z_MAX)

        try:
            old_computed_angles = np.copy(computed_angles)
            
            target_position = np.array([next_x, next_y, next_z])
            target_orientation = np.array([
                [-1.0,  0.0,  0.0],
                [ 0.0,  1.0,  0.0],
                [ 0.0,  0.0, -1.0]
            ])
            target_frame = np.eye(4)
            target_frame[:3, :3] = target_orientation
            target_frame[:3, 3] = target_position
            
            computed_angles = dofbot_chain.inverse_kinematics_frame(target_frame, initial_position=old_computed_angles)
            
            # 答え合わせ（FK）
            real_frame = dofbot_chain.forward_kinematics(computed_angles)
            real_position = real_frame[:3, 3]
            distance_error = np.linalg.norm(real_position - target_position)
            
            new_angles = [90.0 + np.degrees(ang) for ang in computed_angles[1:6]]
            MAX_JUMP = 30.0
            jump_ok = True
            for i in range(5):
                if abs(new_angles[i] - last_servo_angles[i]) > MAX_JUMP:
                    jump_ok = False
                    break
            
            # 【判定】実現可能なら正式採用
            if distance_error < 0.015 and jump_ok:
                target_x, target_y, target_z = next_x, next_y, next_z
                
                # ★修正のポイント：最新の角度を「1つ前の記憶」に移してから、最新を更新する
                prev_servo_angles = list(last_servo_angles)
                last_servo_angles = new_angles
                
                print(f"現在の位置 ➔ X: {target_x*100:.1f}cm, Y: {target_y*100:.1f}cm, Z: {target_z*100:.1f}cm")
                print(f"最新の角度 ➔ ID1:{last_servo_angles[0]:.1f}°, ID2:{last_servo_angles[1]:.1f}°, ID3:{last_servo_angles[2]:.1f}°, ID4:{last_servo_angles[3]:.1f}°, ID5:{last_servo_angles[4]:.1f}°")
                print(f"１歩前の角度 ➔ ID1:{prev_servo_angles[0]:.1f}°, ID2:{prev_servo_angles[1]:.1f}°, ID3:{prev_servo_angles[2]:.1f}°, ID4:{prev_servo_angles[3]:.1f}°, ID5:{prev_servo_angles[4]:.1f}°")
                
                Arm.Arm_serial_servo_write6(
                    last_servo_angles[0], last_servo_angles[1], last_servo_angles[2], 
                    last_servo_angles[3], last_servo_angles[4], 90, DURATION
                )
                time.sleep(WAIT_TIME)
            else:
                computed_angles = old_computed_angles 
                print(f"🛑 [境界線キープ] 危険を検知したため停止しました。")

        except Exception as e:
            print(f"計算エラー: {e}")

    key_receiver.value = ''

# =====================================================================
# 6. 起動時の自動ポジショニング と 表示
# =====================================================================
key_receiver.observe(handle_text_change, names='value')
print("【1歩前記憶機能つき WASD+RFモード】監視スタート")
print(f"➔ 実機アームを初期位置 [後ろ側] (X:{target_x*100:.1f}cm, Z:{target_z*100:.1f}cm) に移動中...")

try:
    initial_angles = cal_servo_angles(target_x, target_y, target_z, base_angles=[0.0]*7)
    last_servo_angles = [initial_angles[1], initial_angles[2], initial_angles[3], initial_angles[4], initial_angles[5]]
    # 起動時は「1つ前の角度」も初期位置と同じにしておきます
    prev_servo_angles = list(last_servo_angles)
    
    Arm.Arm_serial_servo_write6(
        last_servo_angles[0], last_servo_angles[1], last_servo_angles[2], 
        last_servo_angles[3], last_servo_angles[4], 90, 1500
    )
    time.sleep(1.5)
    print("➔ 連動完了！安全に操作できます。")
except Exception as e:
    print(f"初期移動エラー: {e}")

display(app_b)

【1歩前記憶機能つき WASD+RFモード】監視スタート
➔ 実機アームを初期位置 [後ろ側] (X:-15.0cm, Z:25.0cm) に移動中...
➔ 連動完了！安全に操作できます。
